In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, recall_score, balanced_accuracy_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE, SelectFromModel, mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from skrebate import ReliefF
import warnings
from collections import defaultdict, Counter
warnings.filterwarnings('ignore')

# ======================== User Configuration ========================
# Set file paths, analysis conditions, and feature columns here
DATA_PATH_COUPLING = "path_to_coupling_data.xlsx"   # Path to first data file
DATA_PATH_TRADITIONAL = "path_to_traditional_data.xlsx"  # Path to second data file
RAW_ID_COL = 'Dataset'          # Column used for merging subject IDs
ANALYSIS_CONDITION = 'EC'       # Experimental condition to filter
BAND_FILTER = 'Alpha'           # Specific band to filter (e.g., 'Alpha'), set to None if not needed
K_FEATURES = 2                  # Fixed number of features to select
OUTER_FOLDS = 5
INNER_FOLDS = 3

# User-defined feature column names (replace with your actual feature list)
FEATURE_COLUMNS = [
    'Feature_1', 'Feature_2', 'Feature_3',  # example features, please replace
    # add more features as needed...
]
# ===================================================================

class FeatureSelector(BaseEstimator, TransformerMixin):
    """General feature selector supporting multiple methods."""
    def __init__(self, method='random_forest', k=10, feature_names=None):
        self.method = method
        self.k = k
        self.feature_names = feature_names
        self.selected_indices_ = None
        self._selector = None

    def fit(self, X, y=None):
        n_features = X.shape[1]
        self.k = min(self.k, n_features)

        if self.method == 'rfe':
            estimator = SVC(kernel='linear', random_state=42)
            self._selector = RFE(estimator=estimator, n_features_to_select=self.k, step=1)
            self._selector.fit(X, y)
            self.selected_indices_ = np.where(self._selector.support_)[0]

        elif self.method == 'mrmr':
            std = np.std(X, axis=0)
            valid_mask = std > 1e-10
            if not valid_mask.all():
                X_filtered = X[:, valid_mask]
                original_indices = np.where(valid_mask)[0]
            else:
                X_filtered = X
                original_indices = np.arange(n_features)

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_filtered)

            try:
                rel = mutual_info_classif(X_scaled, y, random_state=42)
            except Exception:
                rel, _ = f_classif(X_scaled, y)

            corr = np.abs(np.corrcoef(X_scaled.T))
            np.fill_diagonal(corr, 0)

            rel_norm = rel / (rel.max() if rel.max() > 0 else 1)
            corr_norm = corr / (corr.max() if corr.max() > 0 else 1)

            selected = []
            remaining = list(range(X_filtered.shape[1]))
            for _ in range(self.k):
                best_score = -np.inf
                best_idx = None
                for i in remaining:
                    red_mean = corr_norm[i, selected].mean() if selected else 0
                    score = rel_norm[i] - red_mean
                    if score > best_score:
                        best_score = score
                        best_idx = i
                if best_idx is not None:
                    selected.append(best_idx)
                    remaining.remove(best_idx)

            self.selected_indices_ = original_indices[np.array(selected)]

        elif self.method == 'relieff':
            n_neighbors = min(10, max(1, len(y) // 3))
            self._selector = ReliefF(n_neighbors=n_neighbors, n_features_to_select=self.k, random_state=42)
            self._selector.fit(X, y)
            self.selected_indices_ = self._selector.top_features_[:self.k]

        elif self.method == 'lasso':
            estimator = LogisticRegression(penalty='l1', solver='liblinear', random_state=42, max_iter=1000)
            self._selector = SelectFromModel(estimator=estimator, max_features=self.k, threshold=-np.inf)
            self._selector.fit(X, y)
            self.selected_indices_ = self._selector.get_support(indices=True)

        elif self.method == 'random_forest':
            estimator = RandomForestClassifier(n_estimators=100, random_state=42)
            self._selector = SelectFromModel(estimator=estimator, max_features=self.k, threshold=-np.inf)
            self._selector.fit(X, y)
            self.selected_indices_ = self._selector.get_support(indices=True)

        else:
            raise ValueError("Unknown method")
        return self

    def transform(self, X):
        return X[:, self.selected_indices_]


def load_and_merge_data(path1, path2, id_col, feature_cols):
    """
    Load and merge two Excel files. Returns a merged DataFrame.
    If files are missing, generate synthetic data aligned with feature_cols.
    """
    try:
        data1 = pd.read_excel(path1)
        data2 = pd.read_excel(path2)

        print(f"Data 1 loaded: {data1.shape}")
        print(f"Data 2 loaded: {data2.shape}")

        def clean_subject_id(raw_id):
            s = str(raw_id)
            s = s.replace('.set', '')
            return s.split('_')[0]

        data1['Cleaned_ID'] = data1[id_col].apply(clean_subject_id)
        data2['Cleaned_ID'] = data2[id_col].apply(clean_subject_id)

        merged = pd.merge(data1, data2, on='Cleaned_ID', how='inner', suffixes=('', '_trad'))
        print(f"Merged shape: {merged.shape}")
        return merged

    except Exception as e:
        print(f"Failed to load files: {e}")
        print("\nGenerating synthetic data for demonstration...")
        from sklearn.datasets import make_classification
        n_samples = 100
        n_features = len(feature_cols)
        X_dummy, y_dummy = make_classification(n_samples=n_samples, n_features=n_features,
                                               n_informative=min(5, n_features), random_state=42)
        data = pd.DataFrame(X_dummy, columns=feature_cols)
        data['Group'] = ['MDD' if i == 1 else 'HC' for i in y_dummy]
        data['Condition'] = 'EC'
        data['Band'] = 'Alpha'
        data['Cleaned_ID'] = range(n_samples)
        return data


def create_feature_set(data, condition, feature_columns, band_filter=None):
    """
    Filter data by condition and optional band, then extract feature matrix X and labels y.
    Returns X, y, and the list of feature names actually used.
    """
    subset = data[data['Condition'] == condition].copy()

    if band_filter and 'Band' in subset.columns:
        subset = subset[subset['Band'] == band_filter].copy()

    if subset.empty:
        print("Warning: Filtered dataset is empty, check condition or data.")
        return None, None, None

    available_cols = [c for c in feature_columns if c in subset.columns]
    missing = set(feature_columns) - set(available_cols)
    if missing:
        print(f"Warning: The following columns are missing and will be ignored: {missing}")

    X = subset[available_cols].values
    y = (subset['Group'] == 'MDD').astype(int).values
    return X, y, available_cols


def calculate_specificity(y_true, y_pred):
    """Calculate specificity."""
    from sklearn.metrics import confusion_matrix
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0


# Scoring metrics
scoring = {
    'balanced_acc': 'balanced_accuracy',
    'accuracy': 'accuracy',
    'sensitivity': make_scorer(recall_score),
    'specificity': make_scorer(calculate_specificity),
    'roc_auc': 'roc_auc'
}

# ====================== Main Analysis Pipeline ======================
if __name__ == "__main__":
    # Load and merge data
    data = load_and_merge_data(DATA_PATH_COUPLING, DATA_PATH_TRADITIONAL, RAW_ID_COL, FEATURE_COLUMNS)

    # Extract features and labels
    X, y, current_feature_names = create_feature_set(
        data, ANALYSIS_CONDITION, FEATURE_COLUMNS, band_filter=BAND_FILTER
    )
    if X is None:
        raise ValueError("No valid data could be extracted. Please check configuration.")

    print(f"\nNumber of features used: {len(current_feature_names)}")
    print(f"Sample size: {X.shape[0]}, MDD ratio: {y.mean():.2%}")

    # Feature selection algorithms to compare
    algorithms_to_test = ['rfe', 'mrmr', 'relieff', 'lasso', 'random_forest']

    outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=42)
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=42)
    k_features_range = [K_FEATURES]

    all_results = []
    feature_selection_records = defaultdict(list)
    best_params_records = defaultdict(list)
    best_estimators_records = defaultdict(list)

    print(f"\nFixed number of features to select: {K_FEATURES}")
    print("=" * 60)

    for algo_name in algorithms_to_test:
        print(f"Processing: {algo_name.upper()}")

        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
            ('selector', FeatureSelector(method=algo_name, k=K_FEATURES, feature_names=current_feature_names)),
            ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
        ])

        param_grid = {
            'selector__k': k_features_range,
            'classifier__n_estimators': [100, 150, 200],
            'classifier__max_depth': [2, 5, 10, None],
            'classifier__min_samples_split': [2, 5, 10],
            'classifier__min_samples_leaf': [2, 5, 10],
            'classifier__max_features': ['sqrt', 'log2'],
            'classifier__class_weight': ['balanced']
        }

        grid = GridSearchCV(pipeline, param_grid, cv=inner_cv, scoring='balanced_accuracy', n_jobs=-1)
        nested_scores = cross_validate(grid, X, y, cv=outer_cv, scoring=scoring, n_jobs=-1, return_estimator=True)

        result_row = {
            'Algorithm': algo_name,
            'Balanced_Acc': f"{np.mean(nested_scores['test_balanced_acc']):.3f} ± {np.std(nested_scores['test_balanced_acc']):.3f}",
            'Accuracy': f"{np.mean(nested_scores['test_accuracy']):.3f} ± {np.std(nested_scores['test_accuracy']):.3f}",
            'Sensitivity': f"{np.mean(nested_scores['test_sensitivity']):.3f} ± {np.std(nested_scores['test_sensitivity']):.3f}",
            'Specificity': f"{np.mean(nested_scores['test_specificity']):.3f} ± {np.std(nested_scores['test_specificity']):.3f}",
            'AUC': f"{np.mean(nested_scores['test_roc_auc']):.3f} ± {np.std(nested_scores['test_roc_auc']):.3f}",
        }
        all_results.append(result_row)

        for fold_idx, estimator in enumerate(nested_scores['estimator']):
            best_params_records[algo_name].append(estimator.best_params_)
            best_estimators_records[algo_name].append(estimator.best_estimator_)
            selected_idx = estimator.best_estimator_.named_steps['selector'].selected_indices_
            selected_names = [current_feature_names[i] for i in selected_idx]
            feature_selection_records[algo_name].extend(selected_names)

    # Compile results
    results_df = pd.DataFrame(all_results)
    column_order = ['Algorithm', 'Balanced_Acc', 'Accuracy', 'Sensitivity', 'Specificity', 'AUC']
    results_df = results_df[column_order]

    # Sort by Balanced Accuracy
    def extract_mean(val_str):
        try:
            return float(val_str.split('±')[0])
        except:
            return 0
    results_df['Sort_Key'] = results_df['Balanced_Acc'].apply(extract_mean)
    results_df = results_df.sort_values('Sort_Key', ascending=False).drop('Sort_Key', axis=1)

    print("\n" + "=" * 60)
    print("Final Results (Mean ± Std)")
    print("=" * 60)
    print(results_df.to_string(index=False))

    # Best algorithm details
    best_algo = results_df.iloc[0]['Algorithm']
    print(f"\nBest Algorithm: {best_algo.upper()}")
    param_list = [str(p) for p in best_params_records[best_algo]]
    most_common_param = Counter(param_list).most_common(1)[0][0]
    print(f"Most common best parameters: {most_common_param}")

    # Feature selection frequency
    cnt = Counter(feature_selection_records[best_algo])
    most_common_features = cnt.most_common(10)
    print("\nFeature selection frequency (Top 10):")
    for feat_name, count in most_common_features:
        print(f"  {count:2d} | {feat_name}")

    # Average feature importance
    feature_importance_dict = defaultdict(float)
    for pipe in best_estimators_records[best_algo]:
        selector = pipe.named_steps['selector']
        selected_idx = selector.selected_indices_
        selected_feat_names = [current_feature_names[i] for i in selected_idx]
        rf = pipe.named_steps['classifier']
        importances = rf.feature_importances_
        for name, imp in zip(selected_feat_names, importances):
            feature_importance_dict[name] += imp

    n_folds = len(best_estimators_records[best_algo])
    for key in feature_importance_dict:
        feature_importance_dict[key] /= n_folds

    sorted_importances = sorted(feature_importance_dict.items(), key=lambda x: x[1], reverse=True)
    print("\nAverage feature importance (based on best model):")
    for rank, (feat_name, imp_value) in enumerate(sorted_importances, 1):
        print(f"  Rank {rank:2d} | {imp_value:.4f} | {feat_name}")